In [2]:
import os
import time
import pickle

import pandas as pd
import numpy as np
import torch

import requests
import json
from bs4 import BeautifulSoup
from urllib.parse import quote
from lxml import etree


from chembl_webresource_client.new_client import new_client
from bioservices import UniProt

import torch
from sentence_transformers import SentenceTransformer

/Users/vlad/Documents/University/Master-MIND/DALAS-Project/.venv/lib/python3.11/site-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version
/Users/vlad/Documents/University/Master-MIND/DALAS-Project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# MeSH
MESH_URL = "https://id.nlm.nih.gov/mesh/sparql"

# ChEMBL 
molecule = new_client.molecule
drug_indication = new_client.drug_indication
target = new_client.target
mechanism = new_client.mechanism

# activity = new_client.activity # TODO: get data from there as well

# Open Targets 
OT_URL = "https://api.platform.opentargets.org/api/v4/graphql"

# Clinical Trials
NCT_URL = "https://clinicaltrials.gov/api/v2/studies"

# Sentence Transformer
ST_MODEL = "neuml/pubmedbert-base-embeddings"

# Uniprot for ID mapping
up = UniProt()


# MeSH IDs

Get identificators of diseases of a given family - scrape them from NIH website.

In [3]:
# SCRAPING VERSION: (changed for more efficient and correct RDF query)

# MESH_AUTOIMM_URL = "https://www.ncbi.nlm.nih.gov/mesh?Db=mesh&Cmd=DetailsSearch&Term=%22Autoimmune+Diseases%22%5BMeSH+Terms%5D"
# MESH_OTHER_IMM_URL = "https://www.ncbi.nlm.nih.gov/mesh/68007154"
# # Exctract all MeSH terms of autoimmune diseases
# # Site with MeSH terms with autoimmune diseases
# url =  MESH_AUTOIMM_URL
# response = requests.get(url)
# if response.status_code == 200:
#     html_content = response.text
#     soup = BeautifulSoup(html_content, "html.parser")

# autoimm_a = soup.find("span", string="Autoimmune Diseases").find_all_next("a") # type: ignore

# url =  MESH_OTHER_IMM_URL
# response = requests.get(url)
# if response.status_code == 200:
#     html_content = response.text
#     soup = BeautifulSoup(html_content, "html.parser")

# other_a= soup.find("a", string="Undifferentiated Connective Tissue Diseases").find_all_next("a") # type: ignore 

# mesh_ids = []

# for disease_a in autoimm_a + other_a: 
#     disease_url = disease_a.get('href')
#     if disease_url.startswith("/mesh/"):
#         response = requests.get(f'https://www.ncbi.nlm.nih.gov{disease_url}')
#         if response.status_code == 200:
#             html_content = response.text
#             disease_soup = BeautifulSoup(html_content, "html.parser")
#             mesh_id = disease_soup.find("p", string=lambda text: text and  text.startswith("MeSH Unique ID:")) # type: ignore
#             if mesh_id:
#                 mesh_ids.append(mesh_id.text.split()[-1])

In [4]:
mesh_query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
PREFIX mesh: <http://id.nlm.nih.gov/mesh/>
PREFIX mesh2025: <http://id.nlm.nih.gov/mesh/2025/>
PREFIX mesh2024: <http://id.nlm.nih.gov/mesh/2024/>
PREFIX mesh2023: <http://id.nlm.nih.gov/mesh/2023/>

SELECT ?dis
FROM <http://id.nlm.nih.gov/mesh>
WHERE {  
  ?root rdfs:label "Immune System Diseases"@en .
  ?root meshv:treeNumber ?tn_root .
  ?dis meshv:treeNumber ?tn .
  FILTER(STRSTARTS(STR(?tn),STR(?tn_root))) .
}
"""

response = requests.get(f"{MESH_URL}?query={quote(mesh_query)}")

if not response.ok:
    response.raise_for_status()

root = etree.fromstring(response.text)
ns = {"sr": "http://www.w3.org/2005/sparql-results#"}
mesh_ids = root.xpath("//sr:uri/text()", namespaces=ns)
mesh_ids = [mesh_id.split("/")[-1] for mesh_id in mesh_ids]


# Drugs and Indications

Retrieve raw data about drugs that were tried at least once againt at least one disease of the family.

In [5]:
indications_df = pd.DataFrame(drug_indication.filter(mesh_id__in=mesh_ids))
chembl_ids = pd.unique(indications_df["molecule_chembl_id"]).tolist()
drugs_df = pd.DataFrame(molecule.filter(molecule_chembl_id__in = chembl_ids))

# Drug targets

Get information about all molecular targets of those drugs as well as mechanism of action.

In [6]:
mechanism_df = pd.DataFrame(
    mechanism.filter(
    molecule_chembl_id__in = chembl_ids
    ).only(["molecule_chembl_id",
            "action_type",
            "disease_efficacy",
            "mechanism_of_action", 
            "target_chembl_id"])
)
target_ids = mechanism_df["target_chembl_id"].unique().tolist()
targets_df = pd.DataFrame(
    target.filter(
        target_chembl_id__in = target_ids
        ).only(["target_chembl_id",
                "target_type", 
                "target_components"])
)

targets_df = targets_df[
    targets_df["target_type"].str.contains("PROTEIN") &
    (targets_df["target_components"].apply(len) > 0)
    ]

targets_df["uniprot_ids"] = targets_df["target_components"].apply(
    lambda comps: [ref["xref_id"]
                   for c in comps
                   for ref in c.get("target_component_xrefs", [])
                   if  ref.get("xref_src_db") == "UniProt"]
)

Collapse synonymous UniProt IDs.

In [7]:
up_id_map = up.mapping(fr="UniProtKB", to="UniProtKB", query=list({ x for ids in targets_df["uniprot_ids"] for x in ids})) # type: ignore

100%|██████████| 280/280 [01:42<00:00,  2.73it/s]


In [8]:
primary_up_ids = list({result["to"]["primaryAccession"] for result in up_id_map["results"]})
targets_df["uniprot_ids"] = targets_df["uniprot_ids"].apply(
    lambda up_ids: [up_id for up_id in up_ids if up_id in primary_up_ids]
)

In [9]:
mechanism_df = mechanism_df.merge(targets_df, on="target_chembl_id")

# Disease targets and info

Retrieve disease data as well the list of targets for each disease from OpenTargets database using GraphQL.

In [10]:
nb_top_targets = 10
ot_targets_info = []
indications_df["efo_id"][indications_df["efo_id"].isna()] = ""
for efo_id in indications_df["efo_id"].unique():
  query_string = """
  query disease($efoId: String!, $size: Int!){
      disease(efoId: $efoId) {
      id
      name
      description

      associatedTargets(page: {index: 0, size: $size}, orderByScore: "score desc"){
      rows{
          target {
              id
              proteinIds{
                  id
                  source
              }
          }
          score
        }
      }

      phenotypes {
        rows {
          phenotypeHPO {
            id
            name
            description
          }
        }
      }

  }
  }
  """

  ot_response = requests.post(OT_URL,
                               json={"query": query_string, 
                                     "variables": {"efoId": efo_id.replace(":","_"), 
                                                   "size" : nb_top_targets}})

  if ot_response.status_code == 200:
    ot_targets_info.append(json.loads(ot_response.text)["data"]["disease"])
  else:
    print(ot_response.status_code)
diseases_df = pd.DataFrame([elt for elt in ot_targets_info if elt is not None])

diseases_df["disease_targets"] = diseases_df["associatedTargets"].apply(
    lambda result: [ uniprot["id"]
                    for row in result["rows"]
                    for uniprot in row["target"]["proteinIds"]
                    if uniprot["source"] == 'uniprot_swissprot'
                    ]
)

/var/folders/hr/c9xk46kd0t3c2nlgs6zkg2_h0000gn/T/ipykernel_31994/3369507634.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  indications_df["efo_id"][indications_df["efo_id"].isna()] = ""
/var/folders/hr/c9xk46kd0t3c2nlgs6zkg2_h0000gn/T/i

Get target evidence (abandonned).

In [11]:

# diseases_df["ensembl_ids"] = diseases_df["associatedTargets"].apply(
#     lambda result: [row["target"]["id"]
#                     for row in result["rows"]]
# )
# ensembl_id_map = up.mapping(fr="Ensembl", to="UniProtKB", query=list({ x for ids in diseases_df["ensembl_ids"] for x in ids})) # type: ignore
# uniprot_map = {result["from"]:result["to"]["primaryAccession"] for result in ensembl_id_map["results"]}
# diseases_df["uniprot_ids"] = diseases_df["ensembl_ids"].apply(
#     lambda ensembl_ids: [ uniprot_map[ensembl_id] for ensembl_id in ensembl_ids]
# )
# # get direction of association for each target

# diseases_df["target_evidence"] = [[] for _ in range(diseases_df.shape[0])]
# nb_evidences = 100
# for i, disease in diseases_df.iterrows():
#     query_string = """
#     query disease($efoId: String!, $ensemblIds : [String!]!, $size: Int!){
#     disease(efoId: $efoId) {
#         #id
#         evidences(ensemblIds: $ensemblIds, size: $size){ 
#         count
#         rows{
#             target{
#             id
#             }
#             score
#             datatypeId
#             variantEffect
#             targetModulation
#             targetRole
#             directionOnTrait
#         }
#         }
#     }
#     }
#     """

#     ot_response = requests.post(OT_URL, 
#                                 json={"query": query_string, 
#                                       "variables": {"efoId": disease["id"], 
#                                                     "ensemblIds": disease["ensembl_ids"], 
#                                                     "size": nb_evidences}})

#     if ot_response.status_code == 200:
#         diseases_df.loc[i, "target_evidence"].append(json.loads(ot_response.text)["data"]["disease"]["evidences"]["rows"])
#     else:
#         print(ot_response.status_code)

# Get trial data for indications

Go trhough ClinicalTrials references from drug indications and get trials metadata.

In [12]:
nct_refs = []
indications_df["nct_ids"] = None
for i, indication in indications_df.iterrows():
    for ref in indication["indication_refs"]:
        if ref["ref_type"] == "ClinicalTrials":
            indications_df.at[i,"nct_ids"] = ref["ref_id"].split(",") # type: ignore
            nct_refs.append(ref["ref_id"].split(","))    

all_nct_refs = np.unique([nct_ref for nct_ref_list in nct_refs for nct_ref in nct_ref_list])

nct_info = []
for i in range(0, len(all_nct_refs), 3):
    if i%150==0: print(f"Got {i} out of {len(all_nct_refs)} studies")
    nct_response = requests.get(f"{NCT_URL}?filter.ids={'0%7C'.join(all_nct_refs[i:i+3])}&format=json")
    time.sleep(0.1)  # 0.1 sec pause to respect 10 req/sec
    if nct_response.status_code == 200:
        nct_info.append(nct_response.json()["studies"])
    else:
        print(nct_response.status_code)
trials_df = pd.DataFrame([ct for cts in nct_info for ct in cts])

Got 0 out of 14336 studies
Got 150 out of 14336 studies
Got 300 out of 14336 studies
Got 450 out of 14336 studies
Got 600 out of 14336 studies
Got 750 out of 14336 studies
Got 900 out of 14336 studies
Got 1050 out of 14336 studies
Got 1200 out of 14336 studies
Got 1350 out of 14336 studies
Got 1500 out of 14336 studies
Got 1650 out of 14336 studies
Got 1800 out of 14336 studies
Got 1950 out of 14336 studies
Got 2100 out of 14336 studies
Got 2250 out of 14336 studies
Got 2400 out of 14336 studies
Got 2550 out of 14336 studies
Got 2700 out of 14336 studies
Got 2850 out of 14336 studies
Got 3000 out of 14336 studies
Got 3150 out of 14336 studies
Got 3300 out of 14336 studies
Got 3450 out of 14336 studies
Got 3600 out of 14336 studies
Got 3750 out of 14336 studies
Got 3900 out of 14336 studies
Got 4050 out of 14336 studies
Got 4200 out of 14336 studies
Got 4350 out of 14336 studies
Got 4500 out of 14336 studies
Got 4650 out of 14336 studies
Got 4800 out of 14336 studies
Got 4950 out of 143

# Embeddings

In [13]:
model = SentenceTransformer(ST_MODEL) 

In [ ]:
drug_names = drugs_df["pref_name"].str.lower()
disease_names = diseases_df["name"].str.lower()
# Compute cosine similarities
similarities = model.similarity(model.encode(drug_names), model.encode(disease_names))
name_similarities = [ {"disease_id" : disease["id"], "drug_id" : drug["molecule_chembl_id"], "disease_name":disease["name"].lower(), "drug_name":drug["pref_name"].lower(), "name_similarity" : float(similarities[i,j])} for j, disease in diseases_df.iterrows() for i, drug in drugs_df.iterrows()]
embeddings_df = pd.DataFrame(name_similarities)

# Put All Together

Raw data:
- Drug:
    - `drugs_df`
    - `mechanism_df` 

- Disease:
    - `diseases_df` 

- Drug-Disease:
    - `indications_df` 
    - `trials_df` 
    - `embeddings_df`


## Drugs Data 

In [15]:
# Unpack json, keep interesting variables only
final_drugs_df = (
    drugs_df[
        (drugs_df['availability_type'] > 0) | drugs_df['availability_type'].isna()
        & ~drugs_df['withdrawn_flag'] | drugs_df['withdrawn_flag'].isna()
    ]
    .drop(columns=['atc_classifications', 'availability_type',
                   'cross_references','molecule_hierarchy',
                   'molecule_synonyms','polymer_flag',
                   'usan_stem', 'usan_substem'])
)

final_drugs_df["biotherapeutic"] = final_drugs_df["biotherapeutic"].notnull().astype(int)

chirality_dict = {2:"achiral", 1:"single_enantiomer", 0:"mixture", -1:None}
final_drugs_df["chirality"] = final_drugs_df["chirality"].apply(lambda x: chirality_dict[x])

final_drugs_df = pd.concat([final_drugs_df.reset_index(drop=True), 
                            pd.DataFrame([d if d is not None else {} 
                                          for d in final_drugs_df["molecule_properties"].to_list()]).reset_index(drop=True),
                            pd.DataFrame([d if d is not None else {} 
                                          for d in final_drugs_df["molecule_structures"].to_list()]).reset_index(drop=True)
                            ], axis=1
                        ).drop(columns=["molecule_properties","molecule_structures","molfile"])

final_drugs_df["pref_name"] = final_drugs_df["pref_name"].str.lower()

first_columns = ['molecule_chembl_id','pref_name']
final_drugs_df = final_drugs_df[first_columns + [c for c in final_drugs_df.columns if c not in first_columns]]
final_drugs_df.rename(columns={
    final_drugs_df.columns[0]: "drug_id",
    final_drugs_df.columns[1]: "drug_name"
}, inplace=True)

# Add to each drug a dictionary {action_type : [targets]} 
final_drugs_df["targets"] = None
for i, drug in final_drugs_df.iterrows():
    drug_targets = mechanism_df.loc[mechanism_df["molecule_chembl_id"]== drug["drug_id"],
                                    ["action_type","uniprot_ids"]]
    if drug_targets.shape[0]>0:
        targets_dict = dict()
        for _,r in drug_targets.iterrows():
            action_type = r["action_type"]
            if action_type not in targets_dict:
                targets_dict[action_type] = []
            targets_dict[r["action_type"]] += r["uniprot_ids"]
        final_drugs_df.at[i,"targets"] = targets_dict


## Disease Data

In [30]:
final_diseases_df = diseases_df.rename(columns={"id":"disease_id", "name": "disease_name"})
# TODO: establish correspondence uniprot target id - target evidence - generate "sign" of association
final_diseases_df

,disease_id,disease_name,description,associatedTargets,phenotypes,disease_targets,disease_pathways
0,EFO_0000685,rheumatoid arthritis,"A chronic, systemic autoimmune disorder charac...","{'rows': [{'target': {'id': 'ENSG00000160712',...",{'rows': [{'phenotypeHPO': {'id': 'HP_0001370'...,"[P08887, P29597, P01375, O60674, Q9Y2R2, Q9UM0...","[R-HSA-1059683, R-HSA-110056, R-HSA-112411, R-..."
1,EFO_0000574,lymphoma,A malignant (clonal) proliferation of B- lymph...,"{'rows': [{'target': {'id': 'ENSG00000172936',...",{'rows': [{'phenotypeHPO': {'id': 'HP_0002665'...,"[Q99836, P04637, P10415, P46531, P15056, Q1331...","[R-HSA-1236974, R-HSA-1257604, R-HSA-166058, R..."
2,EFO_0000274,atopic eczema,A common chronic pruritic inflammatory skin di...,"{'rows': [{'target': {'id': 'ENSG00000143631',...",{'rows': [{'phenotypeHPO': {'id': 'HP_0001939'...,"[P20930, P35225, P08887, P04150, P24394, P6294...","[R-HSA-6809371, R-HSA-9725554, R-HSA-6785807, ..."
3,EFO_0001378,multiple myeloma,A bone marrow-based plasma cell neoplasm chara...,"{'rows': [{'target': {'id': 'ENSG00000113851',...",{'rows': [{'phenotypeHPO': {'id': 'HP_0031047'...,"[Q96SW2, P04637, P01116, Q02223, P49917, P0111...","[R-HSA-9679191, R-HSA-111448, R-HSA-139915, R-..."
4,EFO_0004991,Myasthenia gravis,"Myasthenia gravis (MG) is a rare, clinically h...","{'rows': [{'target': {'id': 'ENSG00000171385',...",{'rows': [{'phenotypeHPO': {'id': 'HP_0000651'...,"[Q9UK17, Q9BQ31, Q8NCM2, P22303, Q8TDN2, P2245...","[R-HSA-1296072, R-HSA-5576894, R-HSA-1296072, ..."
...,...,...,...,...,...,...,...
192,EFO_1000491,Primary Effusion Lymphoma,Primary effusion lymphoma (PEL) is a large B-c...,"{'rows': [{'target': {'id': 'ENSG00000171862',...",{'rows': []},"[P60484, P42771, Q8N726, O75626, P40763, Q9994...","[R-HSA-1660499, R-HSA-1855204, R-HSA-199418, R..."
193,MONDO_0017278,autoimmune polyendocrinopathy,A group of diverse conditions that are charact...,"{'rows': [{'target': {'id': 'ENSG00000160224',...",{'rows': []},"[O43918, P13498, P00742, Q9NYK1, Q9NR96, P1785...","[R-HSA-1222556, R-HSA-1236973, R-HSA-3299685, ..."
194,EFO_0009733,psoriasis-related juvenile idiopathic arthritis,Childhood arthritis typically associated with ...,"{'rows': [{'target': {'id': 'ENSG00000232810',...",{'rows': []},"[P01375, Q9NPF7, Q9BYH8, P09769, Q9H939, Q9UMR...","[R-HSA-381340, R-HSA-5357786, R-HSA-5357905, R..."
195,MONDO_0002977,autoimmune disorder of the nervous system,A disorder characterized by the degeneration o...,"{'rows': [{'target': {'id': 'ENSG00000113580',...",{'rows': []},"[P04150, P48551, Q03721, Q9UK17, P22459, Q1280...","[R-HSA-3371497, R-HSA-383280, R-HSA-400253, R-..."


## Drug-Disease Data

### Reactome

In [31]:
drug_targets = np.unique([
    uniprot 
    for targets in final_drugs_df["targets"] 
    if targets
    for uniprots in targets.values() 
    for uniprot in uniprots ])

disease_targets = np.unique([
    uniprot
    for uniprots in final_diseases_df["disease_targets"]
    for uniprot in uniprots
])

all_targets = np.unique(np.concat([drug_targets, disease_targets]))

In [32]:
# TODO: optimize the code
drug_react_map = []
for i in range(0,len(drug_targets),20):
    res = up.mapping(fr="UniProtKB_AC-ID", to="Reactome", query= drug_targets[i: i+20]) # type: ignore
    drug_react_map += res["results"]
disease_react_map = []
for i in range(0,len(disease_targets),20):
    res = up.mapping(fr="UniProtKB_AC-ID", to="Reactome", query= disease_targets[i: i+20]) # type: ignore
    disease_react_map += res["results"]
drug_react_df = pd.DataFrame(drug_react_map)
disease_react_df = pd.DataFrame(disease_react_map)
react_df = pd.concat([drug_react_df, disease_react_df], axis=0).drop_duplicates()
up_react_map = {}
for i, react in react_df.iterrows():
    if react["from"] not in up_react_map:
        up_react_map[react["from"]] = [react["to"]]
    else:
        up_react_map[react["from"]].append(react["to"])
final_drugs_df["drug_pathways"] = final_drugs_df["targets"].apply(
    lambda targets: [ react for up_list in targets.values() for up in up_list if up in up_react_map for react in up_react_map[up] ] if targets else []
)
final_diseases_df["disease_pathways"] = final_diseases_df["disease_targets"].apply(
    lambda targets: [ react for up in targets if up in up_react_map for react in up_react_map[up] ] if targets else []
)

100%|██████████| 1/1 [00:00<00:00, 48.14it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
 73%|███████▎  | 11/15 [00:07<00:02,  1.50it/s]


KeyboardInterrupt: 

# Save results

In [ ]:
with open("../data/01-result/mesh_ids.pkl","wb") as f:
        pickle.dump(mesh_ids,f)
with open("../data/01-result/drugs_df.pkl","wb") as f:
    pickle.dump(final_drugs_df,f)
with open("../data/01-result/indications_df.pkl","wb") as f:
    pickle.dump(indications_df,f)
with open("../data/01-result/trials_df.pkl","wb") as f:
    pickle.dump(trials_df,f)
with open("../data/01-result/diseases_df.pkl","wb") as f:
    pickle.dump(final_diseases_df,f)
with open("../data/01-result/embeddings_df.pkl","wb") as f:
    pickle.dump(embeddings_df,f)